# Bakas: persist satellite data to Google Drive

Downloads every usable Sentinel-2 date of Laguna de Bay (Jan 2024 to Oct 2026) **once**, straight from AWS Open Data into your Google Drive, as Cloud-Optimized GeoTIFFs. Every file is registered in `bakas.db` (`raster_asset` table) with size and SHA-256, so later notebooks read from Drive instead of re-downloading.

**Before running:** put `bakas.db` in `MyDrive/bakas/` (or upload it when the cell below asks).

- One-time run: about 40 to 60 minutes, about 3 to 4 GB of Drive space.
- **Resumable:** if Colab disconnects, run again; finished files are skipped.

In [ ]:
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install pystac-client rasterio
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/bakas"
else:
    ROOT = os.environ.get("BAKAS_ROOT", "./bakas_drive")
os.makedirs(ROOT, exist_ok=True)
DB = f"{ROOT}/bakas.db"
if not os.path.exists(DB):
    if IN_COLAB:
        from google.colab import files
        print("Upload bakas.db"); up = files.upload(); os.replace(next(iter(up)), DB)
    else:
        raise FileNotFoundError(DB)
LIMIT = int(os.environ.get("BAKAS_LIMIT", "0"))   # 0 = all dates; set e.g. 3 to test
print("Drive folder:", ROOT)

In [ ]:
import sqlite3, hashlib, json, datetime as dt, numpy as np, cv2, rasterio
from rasterio.windows import from_bounds
from rasterio.transform import from_origin
from rasterio.warp import transform_bounds
from rasterio.enums import Resampling
from pystac_client import Client
from concurrent.futures import ThreadPoolExecutor

con = sqlite3.connect(DB); con.execute("PRAGMA foreign_keys=ON")
aoi = con.execute("SELECT min_lon,min_lat,max_lon,max_lat,grid_crs,grid_res_m FROM aoi WHERE code='laguna_de_bay'").fetchone()
CRS, RES = aoi[4], aoi[5]
b = transform_bounds("EPSG:4326", CRS, *aoi[:4])
B = (np.floor(b[0]/RES)*RES, np.floor(b[1]/RES)*RES, np.ceil(b[2]/RES)*RES, np.ceil(b[3]/RES)*RES)
W, H = int((B[2]-B[0])/RES), int((B[3]-B[1])/RES)
TRANSFORM = from_origin(B[0], B[3], RES, RES)
STAC = Client.open("https://earth-search.aws.element84.com/v1")
BANDS = ["blue", "green", "red", "nir", "swir16"]
print(f"grid {W} x {H} @ {RES} m, {CRS}")

def local_path(uri): return uri.replace("drive://bakas", ROOT)

def read_grid(href, resampling):
    with rasterio.open(href) as s:
        win = from_bounds(*B, transform=s.transform)
        return s.read(1, window=win, out_shape=(H, W), boundless=True, fill_value=0, resampling=resampling)

def write_cog(path, arr, dtype, nodata, descriptions=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    arr = arr if arr.ndim == 3 else arr[None]
    tmp = path + ".tmp.tif"
    prof = dict(driver="COG", width=W, height=H, count=arr.shape[0], dtype=dtype, crs=CRS, transform=TRANSFORM,
                nodata=nodata, compress="DEFLATE", predictor=2, blocksize=512, overview_resampling="average")
    with rasterio.open(tmp, "w", **prof) as dst:
        dst.write(arr.astype(dtype))
        if descriptions:
            for i, d in enumerate(descriptions, 1): dst.set_band_description(i, d)
    os.replace(tmp, path)
    h = hashlib.sha256(open(path, "rb").read()).hexdigest()
    return os.path.getsize(path), h

def mark(uri, size, sha):
    con.execute("UPDATE raster_asset SET status='stored', bytes=?, sha256=?, stored_at=? WHERE storage_uri=?",
                (size, sha, dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"), uri))
    con.commit()

## 1. Scene classification (SCL) for every usable date
Small and fast. Used for clouds and to build the lake mask.

In [ ]:
usable = con.execute("SELECT acquisition_id, acquired_date FROM acquisition WHERE aoi_id=1 AND is_usable=1 ORDER BY acquired_date").fetchall()
if LIMIT: usable = usable[:LIMIT]
scene_ids = {a: [r[0] for r in con.execute("SELECT scene_id FROM acquisition_scene WHERE acquisition_id=?", (a,))] for a, _ in usable}
items_by_acq = {}
def items_for(a):
    if a not in items_by_acq:
        items_by_acq[a] = list(STAC.search(collections=["sentinel-2-l2a"], ids=scene_ids[a]).items())
    return items_by_acq[a]
def is_stored(uri):
    return con.execute("SELECT status='stored' FROM raster_asset WHERE storage_uri=?", (uri,)).fetchone()[0] and os.path.exists(local_path(uri))

def fetch_scl(acq_id):
    scl = np.zeros((H, W), np.uint8)
    for it in items_for(acq_id):
        s = read_grid(it.assets["scl"].href, Resampling.nearest)
        scl[(scl == 0) & (s != 0)] = s[(scl == 0) & (s != 0)]
    return scl

todo = [(a, d) for a, d in usable if not is_stored(f"drive://bakas/rasters/laguna_de_bay/{d}/scl.tif")]
print(len(todo), "SCL layers to download")
with ThreadPoolExecutor(8) as ex:
    for (a, d), scl in zip(todo, ex.map(lambda r: fetch_scl(r[0]), todo)):
        uri = f"drive://bakas/rasters/laguna_de_bay/{d}/scl.tif"
        mark(uri, *write_cog(local_path(uri), scl, "uint8", 0, ["scl"])); print(d, "scl stored", flush=True)

## 2. Lake mask
Water in more than 60% of clear observations; `connectedComponentsWithStats` keeps the lake itself.

In [ ]:
water = np.zeros((H, W), np.int32); valid = np.zeros((H, W), np.int32)
for a, d in usable:
    with rasterio.open(local_path(f"drive://bakas/rasters/laguna_de_bay/{d}/scl.tif")) as s: scl = s.read(1)
    v = (scl != 0) & ~np.isin(scl, [1, 3, 8, 9, 10]); water += (scl == 6) & v; valid += v
lake = ((water / np.maximum(valid, 1) > 0.6) & (valid >= min(10, len(usable)))).astype(np.uint8)
n, lab, st, _ = cv2.connectedComponentsWithStats(lake, 8)
lake = (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
uri = "drive://bakas/rasters/laguna_de_bay/static/lake_mask.tif"; mark(uri, *write_cog(local_path(uri), lake, "uint8", 0, ["lake"]))
SHORE = cv2.distanceTransform(lake * 255, cv2.DIST_L2, 5)
KEEP = cv2.dilate(lake, np.ones((101, 101), np.uint8)) > 0     # lake + 1 km of shore; land beyond is nodata (saves ~70% space)
print("lake area km2:", round(lake.sum() * RES * RES / 1e6, 1))

## 3. Surface reflectance (lake + 1 km shoreline)

In [ ]:
def fetch_refl(acq_id):
    refl = np.full((len(BANDS), H, W), -9999, np.int16)
    for it in items_for(acq_id):
        off = 0 if it.properties.get("earthsearch:boa_offset_applied", False) else (
            -1000 if float(it.properties.get("s2:processing_baseline", "0")) >= 4 else 0)
        for k, band in enumerate(BANDS):
            a = read_grid(it.assets[band].href, Resampling.average if band != "swir16" else Resampling.nearest).astype(np.int32)
            sel = (refl[k] == -9999) & (a > 0) & KEEP
            refl[k][sel] = np.clip(a[sel] + off, -9998, 32767)
    return refl

todo = [(a, d) for a, d in usable if not is_stored(f"drive://bakas/rasters/laguna_de_bay/{d}/reflectance.tif")]
print(len(todo), "dates to download")
done = 0
with ThreadPoolExecutor(4) as ex:
    for (a, d), refl in zip(todo, ex.map(lambda r: fetch_refl(r[0]), todo)):
        uri = f"drive://bakas/rasters/laguna_de_bay/{d}/reflectance.tif"
        mark(uri, *write_cog(local_path(uri), refl, "int16", -9999, BANDS))
        done += 1; print(f"{done}/{len(todo)}  {d}  reflectance stored", flush=True)

## 4. Color distance and baseline
Same method as the scan: MNDWI water test, Mahalanobis color distance from open water more than 2 km offshore, per-pixel median as the baseline. Stored as int16 (value x 100).

In [ ]:
def load_date(date):
    with rasterio.open(local_path(f"drive://bakas/rasters/laguna_de_bay/{date}/reflectance.tif")) as s:
        a = s.read().astype(np.float32); a[a == -9999] = np.nan; a *= 1e-4
    with rasterio.open(local_path(f"drive://bakas/rasters/laguna_de_bay/{date}/scl.tif")) as s:
        scl = s.read(1)
    return dict(zip(BANDS, a)), scl

stored = [d for a, d in usable]
def smooth_nan(x, s=1.5):
    m = np.isfinite(x).astype("f4"); v = np.nan_to_num(x).astype("f4")
    o = cv2.GaussianBlur(v, (0, 0), s) / np.maximum(cv2.GaussianBlur(m, (0, 0), s), 1e-3); o[m == 0] = np.nan; return o

def color_distance(bands, scl):
    cloudbuf = cv2.dilate(np.isin(scl, [3, 8, 9, 10]).astype(np.uint8), np.ones((31, 31), np.uint8)) > 0
    r, g, b, sw = [np.clip(np.nan_to_num(bands[k], nan=1e-3), 1e-3, None) for k in ["red", "green", "blue", "swir16"]]
    valid = (lake > 0) & np.isfinite(bands["green"]) & ~cloudbuf & ((g - sw) / (g + sw) > 0) & (SHORE > 2)
    if valid.sum() < 50000: return None
    lg = np.log(g).astype("f4"); mu = cv2.blur(lg, (5, 5)); rough = np.sqrt(np.clip(cv2.blur(lg * lg, (5, 5)) - mu * mu, 0, None))
    bz = (b - np.median(b[valid])) / (np.median(np.abs(b[valid] - np.median(b[valid]))) * 1.4826 + 1e-5)
    ok = valid & (rough <= np.percentile(rough[valid], 95)) & (bz < 4)
    ref = ok & (SHORE > 100)
    if ref.sum() < 20000: ref = ok
    F = np.stack([np.log(r / g), np.log(b / g), np.log(g)], -1)
    m0 = np.median(F[ref], 0); inv = np.linalg.inv(np.cov((F[ref] - m0).T) + np.eye(3) * 1e-6); d = F - m0
    return smooth_nan(np.where(ok, np.sqrt(np.einsum("...i,ij,...j->...", d, inv, d)), np.nan))

def to_i16(x): return np.where(np.isfinite(x), np.clip(np.round(x * 100), -9998, 32767), -9999).astype(np.int16)

Ds = []
for d in stored:
    uri = f"drive://bakas/rasters/laguna_de_bay/{d}/color_distance.tif"
    path = local_path(uri)
    if os.path.exists(path):
        with rasterio.open(path) as s: a = s.read(1).astype("f4"); a[a == -9999] = np.nan; Ds.append(a / 100); continue
    D = color_distance(*load_date(d))
    if D is None: print(d, "too cloudy after masking, skipped"); continue
    mark(uri, *write_cog(path, to_i16(D), "int16", -9999, ["color_distance_x100"])); Ds.append(D)
    print(d, "color distance stored", flush=True)

base = np.full((H, W), np.nan, np.float32)
for r0 in range(0, H, 200):
    base[r0:r0+200] = np.nanmedian(np.stack([x[r0:r0+200] for x in Ds]), 0)
uri = "drive://bakas/rasters/laguna_de_bay/static/baseline.tif"; mark(uri, *write_cog(local_path(uri), to_i16(base), "int16", -9999, ["baseline_x100"]))
print("baseline stored from", len(Ds), "dates")

## 3. Check what is stored

In [ ]:
for row in con.execute('''SELECT product_code, status, COUNT(*), ROUND(SUM(COALESCE(bytes,0))/1e6,1) AS mb
                          FROM raster_asset GROUP BY product_code, status ORDER BY product_code'''):
    print(row)

## Reuse later (no re-download)

```python
import rasterio, sqlite3, numpy as np
ROOT = "/content/drive/MyDrive/bakas"
con = sqlite3.connect(f"{ROOT}/bakas.db")
uri = con.execute("SELECT storage_uri FROM raster_asset r JOIN acquisition a USING(acquisition_id) "
                  "WHERE a.acquired_date='2026-05-06' AND r.product_code='reflectance'").fetchone()[0]
with rasterio.open(uri.replace("drive://bakas", ROOT)) as src:
    blue, green, red, nir, swir16 = src.read().astype("float32") * 1e-4   # nodata = -9999
```